## Homework 4: Post-training

__Soft deadline: 08.12.2026 23:59__ \
__Hard deadline: 10.12.2026 23:59__

### About the assignment

In this assignment, you are going to fine-tune a pretrained LLM, [`Qwen3-0.6B-Base`](https://huggingface.co/Qwen/Qwen3-0.6B-Base), to solve math problems. As in the standard post-training setup, training will happen in two stages: SFT and RL. You will need to implement both stages yourself, using LoRA for fine-tuning.

We expect the whole training cycle to take no more than a day on a single T4.

The use of agents is __strictly prohibited__ in this assignment.

### Grading and penalties

The maximum grade for this assignment is __11.5 points__.

The assignment must be completed independently. "Similar" solutions are considered plagiarism, and all students involved (including those whose work was plagiarized) cannot receive more than 0 points for it. All code must be written by yourself. Using someone else's code is prohibited, even with a reference to the source. Within reason, of course. A couple of obvious lines of code implementing some minor functionality are acceptable.

### About the data

We will use a mixture of two datasets of math problems: [GSM8K](https://huggingface.co/datasets/openai/gsm8k) (human-written problems) and [OpenMathInstruct-2](https://huggingface.co/datasets/nvidia/OpenMathInstruct-2) (generated by Llama-3.1-405B-Instruct in the style of GSM8K). The processed and cleaned dataset is available at [`ashaba1in/sft_grpo_gsm8k_omi2`](https://huggingface.co/datasets/ashaba1in/sft_grpo_gsm8k_omi2).

| Split | Size | Content |
| --- | --- | --- |
| `sft_train` | 21,000 | mixture of GSM8K and OMI-2 |
| `rl_train` | 5,973 | GSM8K with no overlap with SFT |
| `dev` | 500 | GSM8K with no overlap with SFT and RL |
| `test` | 1,319 | GSM8K test split |

The `question` column contains the question with a prompt explaining the task. In `sft_train`, the `answer` column contains a detailed answer with reasoning; for all other splits – only the numerical answer, used to check whether it matches the prediction.

In [3]:
from datasets import load_dataset

dataset = load_dataset("ashaba1in/sft_grpo_gsm8k_omi2")
dataset

DatasetDict({
    sft_train: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 21000
    })
    rl_train: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 5973
    })
    dev: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'question', 'answer'],
        num_rows: 1319
    })
})

In [4]:
example = dataset['sft_train'][0]
print('SFT example\n')
print("Question:\n" + example["question"])
print("\nAnswer:\n" + example["answer"])

SFT example

Question:
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: Emily purchased 7 sets of blue pens and 3 sets of green pens. Each set contains 12 pens. How many more blue pens than green pens did Emily buy?
Solution:

Answer:
 Let's break down the number of blue pens and green pens Emily purchased.
For blue pens, she purchased 7 sets. Each set contains 12 pens, so she purchased 7 * 12 = 84 blue pens.
For green pens, she purchased 3 sets. Each set contains 12 pens, so she purchased 3 * 12 = 36 green pens.
Now we find out how many more blue pens than green pens she bought.
The difference is 84 - 36 = 48.
So Emily bought \boxed{48} more blue pens than green pens.


In [5]:
example = dataset['rl_train'][0]
print('RL example\n')
print("Question:\n" + example["question"])
print("\nAnswer:\n" + example["answer"])

RL example

Question:
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: At A.T. Cross Luxury Pens, a pencil sells at twenty-five cents while a pen sells at fifteen cents. Bowen buys 40 pens and 2/5 times more pencils than pens from A.T. Cross Luxury Pens. Calculate the total amount of money Bowen spends.
Solution:

Answer:
20


In [6]:
fewshot_prefix = """
Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: The result from the 40-item Statistics exam Marion and Ella took already came out. Ella got 4 incorrect answers while Marion got 6 more than half the score of Ella. What is Marion's score?
Solution: Ella's score is 40 items - 4 items = 36 items.
Half of Ella's score is 36 items / 2 = 18 items.
So, Marion's score is 18 items + 6 items = 24 items.
The final answer is \boxed{24}.

Solve the following math problem. Reason step by step and put the final answer in \boxed{}.

Problem: Stephen made 10 round trips up and down a 40,000 foot tall mountain. If he reached 3/4 of the mountain's height on each of his trips, calculate the total distance he covered.
Solution: Up a mountain, Stephen covered 3/4*40000 = 30000 feet.
Coming down, Stephen covered another 30000 feet, making the total distance covered in one round to be 30000+30000 = 60000.
Since Stephen made 10 round trips up and down the mountain, he covered 10*60000 = 600000
The final answer is \boxed{600000}.

"""

### vLLM

[vLLM](https://docs.vllm.ai) is an inference library that generates text much faster than the built-in `model.generate`. The `vllm_utils.py` file below contains helper functions for working with vLLM.
To get familiar with the library, we recommend reading the [vLLM quickstart](https://docs.vllm.ai/en/stable/getting_started/quickstart/#on-attention-backends) and the [vLLM usage examples](https://docs.vllm.ai/en/stable/examples/). You can also read the code itself on [GitHub](https://github.com/vllm-project/vllm/).

Usage pipeline:
1. vLLM loads its own copy of the frozen base model in fp16.
2. The LoRA adapter is the only thing that changes in the whole model. Therefore, before generation, you need to export it with `save_lora_for_vllm` and pass the corresponding path to `generate` together with a __new__ `lora_id` value.
3. For RL, use the merged SFT model (base + LoRA weight delta) as the base model and train a new LoRA adapter.

Practical notes:
- The fraction of GPU memory allocated to vLLM can be controlled with `gpu_memory_utilization`.
- To get several answers for each prompt, use `n=` instead of generating repeatedly. This saves time, since the prompt is processed only once.
- Train the model on the `token_ids` returned by vLLM, not on a re-tokenization of the text, since the new tokenization may differ.
- For the base model, pass `stop=["\nSolve the following math problem."]`, otherwise it may keep writing new problems after the answer.

In [20]:
%%writefile vllm_utils.py

"""
Helper functions fro vLLM with LoRA support

Pipeline:
  * vLLM keeps its own copy of the frozen base weights (fp16) and applies trained LoRA adapter on the fly.
  * Before generation, export the adapter using `save_lora_for_vllm` on disk and
    pass its path to `generate` with a NEW integer `lora_id`.
    vLLM caches adapters by id, so reusing an id keeps generating with this id's weights.
"""

import json
import os

import torch
from safetensors.torch import save_file


def make_llm(
    model_name_or_path: str, gpu_memory_utilization: float = 0.3,
    max_model_len: int = 1024, max_lora_rank: int = 16,
    enable_lora: bool = True, seed: int = 0, enforce_eager: bool = False
):
    """Create a vLLM engine that occupies a specified part of GPU.

    gpu_memory_utilization is the fraction of the whole GPU vLLM can take (weights + KV cache).
    max_lora_rank must be >= trained LoRA rank.
    """
    from vllm import LLM

    return LLM(
        model=model_name_or_path, dtype="float16", gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len, enable_lora=enable_lora, max_lora_rank=max_lora_rank,
        max_loras=1, seed=seed, enforce_eager=enforce_eager
    )


def generate(
    llm, prompts, n: int = 1, temperature: float = 0.0,
    top_p: float = 1.0, max_tokens: int = 512,
    stop=None, seed=None, lora_path=None, lora_id=None
):
    """Sample n completions for every prompt.

    Returns a list (one entry per prompt) of lists (one entry per sample) of dicts:
        {"text": str, "token_ids": list[int], "finish_reason": "stop" | "length"}
    finish_reason == "length" means the completion was truncated at max_tokens.
    Make sure to `token_ids` (not a re-tokenization of `text`) when you compute log-probs for GRPO.
    Pass lora_path=None to sample from the base weights only.
    """
    from vllm import SamplingParams
    from vllm.lora.request import LoRARequest

    if temperature == 0.0:
        assert n == 1, "greedy decoding with n > 1 returns n identical samples"

    params = SamplingParams(
        n=n, temperature=temperature, top_p=top_p,
        max_tokens=max_tokens, stop=stop, seed=seed
    )
    lora_request = None
    if lora_path is not None:
        assert lora_id is not None and lora_id >= 1, "use a new positive integer id for every new adapter"
        lora_request = LoRARequest(f"adapter-{lora_id}", lora_id, lora_path)

    outputs = llm.generate(prompts, params, lora_request=lora_request, use_tqdm=False)
    return [[
        {"text": c.text, "token_ids": list(c.token_ids), "finish_reason": c.finish_reason}
        for c in out.outputs
    ] for out in outputs]


def save_lora_for_vllm(
    lora_weights: dict, out_dir: str, scaling: float, base_model_name_or_path: str = ""
):
    """Export LoRA weights in the adapter format vLLM reads.

    lora_weights: {module_name: (A, B)} for every linear layer you adapted, where
        module_name is the layer's name in model.named_modules(), e.g. "model.layers.0.self_attn.q_proj",
        A has shape [r, in_features] and B has shape [out_features, r],
        so that the adapted layer computes  y = x W^T + scaling * (x A^T) B^T.
    scaling: the factor your LoRA applies to B @ A (usually alpha / r). It is folded into B here.
    """

    os.makedirs(out_dir, exist_ok=True)
    ranks = {tuple(A.shape)[0] for A, _ in lora_weights.values()}
    assert len(ranks) == 1, f"all adapters must have the same rank, got {ranks}"

    r = ranks.pop()
    tensors, targets = {}, set()
    for name, (A, B) in lora_weights.items():
        assert A.dim() == 2 and B.dim() == 2 and A.shape[0] == r and B.shape[1] == r, (name, A.shape, B.shape)
        prefix = f"base_model.model.{name}"
        tensors[f"{prefix}.lora_A.weight"] = A.detach().float().to(torch.float16).cpu().contiguous()
        tensors[f"{prefix}.lora_B.weight"] = (B.detach().float() * scaling).to(torch.float16).cpu().contiguous()
        targets.add(name.split(".")[-1])

    save_file(tensors, os.path.join(out_dir, "adapter_model.safetensors"))
    config = {
        "peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": base_model_name_or_path,
        "r": r, "lora_alpha": r,  # alpha / r = 1 because `scaling` is already folded into B
        "target_modules": sorted(targets), "lora_dropout": 0.0, "bias": "none",
        "fan_in_fan_out": False, "use_rslora": False, "use_dora": False,
        "modules_to_save": None, "inference_mode": True,
    }

    with open(os.path.join(out_dir, "adapter_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    return out_dir

Overwriting vllm_utils.py


In [ ]:
# Run this once to check that vLLM works on your GPU.

from vllm_utils import make_llm, generate, save_lora_for_vllm

llm = make_llm("Qwen/Qwen3-0.6B-Base", gpu_memory_utilization=0.8)

dev = dataset['dev']

outs = generate(
    llm, [fewshot_prefix + ex for ex in dev[:4]['question']],
    n=2, temperature=0.6, top_p=0.95,
    max_tokens=512, stop=["\nSolve the following math problem."])

for question, answer, samples in zip(dev[:4]["question"], dev[:4]["answer"], outs):
    print("Question:\n" + question)
    print("\nGold answer: " + answer)
    print("\nOutputs:")
    for s in samples:
        print(f'  [Finish reason "{s['finish_reason']}", {len(s['token_ids'])} tokens] {s['text'][-150:]!r}')
    print('=' * 100)

# Before every generation:
#   lora = {name: (A, B) for every adapted linear layer}    # A: [r, in_features], B: [out_features, r]
#   save_lora_for_vllm(lora, "adapters/current", scaling=alpha / r)
#   outs = generate(
#       llm, prompts, n=8, temperature=1.0, max_tokens=512,
#       lora_path="adapters/current", lora_id=step + 1
#   )

### Quality evaluation

To evaluate quality, we will generate answers for the test set with vLLM and measure the accuracy of the predictions. For this, we need to implement several functions for processing model outputs. They will also be used later for RL.

__Task 1 (0.5 points).__ Implement the functionality for processing answers.

- `extract_answer(text)`: extracts the contents of the __last__ `\boxed{...}`, normalizing the text inside: all extra curly braces, commas, dollar and percent signs (`\%` and `%`), spaces and trailing periods are removed. If a number remains after normalization, it is returned as a `float`; otherwise, `None` is returned.

- `is_correct(pred, gold)`: checks that `pred` is not `None` and equals `float(gold)` up to a tolerance of `1e-4`.

- `format_ok(text, finish_reason)`: checks whether the output has the correct format: the generation must end with EOS (`finish_reason == "stop"` from vLLM), contain exactly one `\boxed{`, and `extract_answer` must return a number.

- `is_repetitive(text)`: returns `True` if there exist 10 consecutive words that occur 3 or more times.

- `pass_at_k(n, c, k)`: the unbiased estimator from [Chen et al. (2021)](https://arxiv.org/abs/2107.03374). If $c$ out of $n$ answers for a problem are correct, then $\text{pass@}k = 1 - \binom{n-c}{k} \big/ \binom{n}{k}$, averaged over problems. Here $n \ge k$.

In [ ]:
# your code here

In [ ]:
# Do not change the tests.

# extract_answer
assert extract_answer("She has 3 + 4 = 7 apples. The final answer is \\boxed{7}.") == 7
assert extract_answer("\\boxed{1,000}") == 1000
assert extract_answer("The answer is \\boxed{\\$42.50}.") == 42.5
assert extract_answer("\\boxed{50\\%}") == 50
assert extract_answer("\\boxed{-3}") == -3
assert extract_answer("\\boxed{ 12 }") == 12
assert extract_answer("First \\boxed{3}, then \\boxed{5}") == 5  # only the last box counts
assert extract_answer("\\boxed{18 \\text{ dollars}}") is None
assert extract_answer("\\boxed{\\frac{1}{2}}") is None           # GSM8K answers are plain numbers
assert extract_answer("The answer is 7.") is None
assert extract_answer("\\boxed{}") is None
assert extract_answer("\\boxed{7") is None

# is_correct
assert is_correct(extract_answer("\\boxed{72}"), "72")
assert is_correct(extract_answer("\\boxed{72.0}"), "72")
assert not is_correct(extract_answer("\\boxed{71}"), "72")
assert not is_correct(None, "72")

# format_ok
assert format_ok("So the total is 7. The final answer is \\boxed{7}.", finish_reason="stop")
assert not format_ok("So the total is 7. The final answer is \\boxed{7}.", finish_reason="length")
assert not format_ok("It is \\boxed{3} or \\boxed{4}.", finish_reason="stop")
assert not format_ok("The answer is 7.", finish_reason="stop")
assert not format_ok("The answer is \\boxed{seven}.", finish_reason="stop")

# is_repetitive
assert is_repetitive("so " * 40)
assert is_repetitive("Then we add 2 to the total and get the new total. " * 3)
assert not is_repetitive("Natalia sold 48 / 2 = 24 clips in May. She sold 48 + 24 = 72 clips in total.")
assert not is_repetitive("Then we add 2 to the total and get the new total. " * 2)

# pass_at_k
assert abs(pass_at_k(n=5, c=2, k=1) - 0.4) < 1e-9
assert abs(pass_at_k(n=10, c=3, k=2) - (1 - 21 / 45)) < 1e-9
assert pass_at_k(n=5, c=0, k=3) == 0.0
assert pass_at_k(n=5, c=4, k=2) == 1.0
print("All Task 1 tests passed")

__Task 2 (0.5 points).__ Implement an `evaluate(...)` function that takes a vLLM engine, a list of problems, a prompt prefix, stop strings and, optionally, a LoRA adapter, and returns:

- __greedy accuracy__ (temperature 0): the accuracy ($p$) of the answers with a 95% [confidence interval](https://en.wikipedia.org/wiki/Binomial_proportion_confidence_interval) $p \pm 1.96\sqrt{p(1-p)/N}$.

- __mean pass@1 over 4 samples__ (temperature 0.6, top-p 0.95): the mean accuracy of 4 answers per problem.

- __format compliance__, __repetition rate__, __truncation rate__ (`finish_reason == "length"`) and __mean response length__ in tokens, on the _greedy_ generations.

Use `max_tokens=512` everywhere.

In [ ]:
# your code here

__Task 3 (0.5 points).__ Evaluate the base model on the test data `dataset["test"]` in zero-shot and few-shot modes (prefix `fewshot_prefix`, `stop=["\nSolve the following math problem."]`). Analyze the results. What conclusions can you draw?

In [ ]:
# your code here

### Supervised Fine-tuning

__Task 4 (2 points).__ Train a LoRA adapter for `Qwen3-0.6B-Base` on the SFT split for one epoch. During training, log the loss, train and validation accuracy, format compliance of the generations, and the fraction of repetitive texts. Compare the results of the SFT model with the base model from __Task 3__. Draw conclusions. For the RL stage, we need a model that follows the format but is still diverse enough in its generations (RL needs good exploration). Did you manage to achieve this?

In [ ]:
# your code here

### Group Relative Policy Optimization

#### Method description
For each prompt $q$, we generate a group of $G$ responses $o_1, \dots, o_G$ with the current policy and compute a reward $r_i \in \{0, 1\}$ for each of them: 1 if the answer is correct and has the required format, 0 otherwise. The advantage is computed from the rewards within the group:

$$A_i = \frac{r_i - \mathrm{mean}(r_1, \dots, r_G)}{\mathrm{std}(r_1, \dots, r_G)}.$$

During training, the following objective is minimized 
<!-- All tokens of $o_i$ share the advantage $\hat A_i$. Using the probability ratio $\rho_{i,t} = \pi_\theta(o_{i,t} \mid q, o_{i,<t}) \,/\, \pi_{\theta_\text{old}}(o_{i,t} \mid q, o_{i,<t})$, the loss over a batch of completions is (note the [DAPO](https://arxiv.org/abs/2503.14476)-style normalization) -->

$$\mathcal{L}(\theta) = -\frac{1}{\sum_i |o_i|} \sum_i \sum_{t=1}^{|o_i|} \Big[ \min\Big(\rho_{i,t} A_i,\; \mathrm{clip}(\rho_{i,t}, 1-\varepsilon, 1+\varepsilon)\, A_i\Big) - \beta D_{i,t} \Big],$$

where $\rho_{i,t} = \pi_\theta(o_{i,t} \mid q, o_{i,<t}) \,/\, \pi_{\theta_\text{old}}(o_{i,t} \mid q, o_{i,<t})$ is the ratio between the policy being optimized and the policy used to generate the responses. $D_{i,t}$ is an unbiased per-token Monte Carlo estimate of the KL divergence $\mathrm{KL}(\pi_\theta \,\|\, \pi_\text{ref})$ with respect to the SFT policy:

$$D_{i,t} = \frac{\pi_\text{ref}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - \log \frac{\pi_\text{ref}(o_{i,t} \mid q, o_{i,<t})}{\pi_\theta(o_{i,t} \mid q, o_{i,<t})} - 1.$$

Note that we depart from the original [GRPO](https://arxiv.org/pdf/2501.12948) formulation and normalize the loss by the total number of generated tokens. The original GRPO averages the loss over each response separately. [Liu et al. (2025)](https://arxiv.org/abs/2503.20783) show that such normalization encourages excessive lengthening of incorrect responses.

__Task 5 (4 points).__ Implement GRPO and use it to fine-tune a LoRA adapter for the SFT model on the corresponding data (`rl_train`). During training, track the mean reward, the fraction of fully correct and fully incorrect groups, the fraction of responses in the correct format, the KL divergence, the mean accuracy and the response length. Compare the GRPO model with the base and SFT models, as in the previous task. Compare answer accuracy as $k$ in `pass@k` increases ($k = 1, 2, 4, \dots, 64$). Give an example of a prompt for which greedy generation of the GRPO model gives the correct answer while the base model gives a wrong one, and vice versa. Draw conclusions.

__Important:__ 
1. The chosen way of computing the advantage zeroes out the gradient for groups in which all answers are correct or all are wrong. So, to save time, discard such groups.
2. RL may diverge if the batch is not large enough. We recommend choosing a group size of $G = 8$ and optimizing over at least $B = 8$ prompts at each step.
3. Pay attention to the temperature and the parameter $\beta$. Think about what role they play and what values they should take.

In [ ]:
# your code here

### Assessing the importance of components

Once we have a working solution, it is worth figuring out what actually matters for it. To do this, we will run a series of ablations. Each item is worth __1 point__.

1. Train GRPO directly from the base model, skipping SFT. How does this modification affect convergence speed and final quality?
2. Compare the model trained with your value of $\beta$ and a model with $\beta = 0$.
3. Try removing the std normalization in the advantage, as recommended in [Dr. GRPO](https://arxiv.org/abs/2503.20783). What changed?
4. Train a rank-1 LoRA for GRPO and compare it with your chosen rank. Did it get worse?

In [ ]:
# your code here